# v8.7 — score-stationary decode inner loop — the gate (CUDA-core, Colab T4)

Four measured negatives (Cut 2 tensor cores, v8.5 double-buffer, v8.6 occupancy + key-ILP) all failed to
move decode off ~10% HBM → the floor is the **per-key warp-shuffle reduction + serial online-softmax
recurrence**, and it is **not hideable**. v8.7 stops hiding and **removes** it with a score-stationary
relayout (the FlashDecoding inner loop):

- **lane = key** (not head-dim): lane `l` computes the FULL dot product q·k_c → the score lives in its
  own register, **no per-key cross-lane reduction**.
- **softmax once per 32-key group** (one `warp_reduce_max` + one `warp_reduce_sum`) → the serial
  recurrence shortens **32×**.
- **PV transpose** via single-hop `__shfl` broadcasts of `p_c` — independent, they pipeline.
- **FP16 transposed smem** (~18 KB → ~3 blocks/SM) holds occupancy fixed (v8.6 proved FP16 smem is
  perf-neutral), so the inner-loop **layout is the only variable** vs Cut 1.

**Prediction:** roofline still BLIND (same `AI=2G/b` floor). Mechanistically v8.7 should finally **drop
µs/tok** — strongest at **d=64**; **d=128 is at risk** (full-D smem reads per key may flip it to
smem-BW-bound). **Counter-prediction:** if µs/tok drops but %HBM stays ~10%, the new floor is per-CTA
**load** latency → v9 FP8 (capacity) is the right next lever. Gate = `v8_gqa_ss` correct (+ Cut 1 not
regressed) → the 3-way A/B reads whether removing the reduction moved the floor.

## 0. Dependencies + GPU (venv-safe)

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes --
#    importing torch first on a numpy-less venv (vast.ai) prints 'Failed to initialize NumPy'.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell (the old CPU torch stays loaded until restart).')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Roofline — UNCHANGED (`AI=2G/b`, same floor); the deliverable is µs/tok

The model can't see the inner-loop layout, so it predicts the **same** HBM floor as Cut 1 / occ / ilp.
The measured µs/tok drop (and whether %HBM finally climbs off ~10%) is the whole result.

In [ ]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_75')
print('arch:', arch.name, '| HBM', arch.hbm_bw_gbps, 'GB/s')
print(f"{'G':>3} | {'AI=2G/b':>8} | {'limiter':>7} | {'t_hbm floor':>12}")
for G in (1,2,4,8,16,32):
    e = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=G)
    print(f'{G:>3} | {e.arithmetic_intensity:8.1f} | {e.limiter.upper():>7} | {e.t_hbm*1e3:9.4f}ms')
print('\nRoofline is BLIND: identical for v8_gqa / v8_gqa_occ / v8_gqa_ss.')
print('Prediction: ss drops us/tok (best at d=64); d=128 at risk of smem-BW; %HBM may stay ~10%.')


## 3. Build v8_gqa_ss (JIT) — watch the ptxas reg/smem line (R3/R4)

In [ ]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v8_gqa_ss')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
ss = build_kernel('v8_gqa_ss'); print('built v8.7 (score-stationary):', ss)


## 4. Correctness gate — v8_gqa_ss + Cut 1 regression (Gate 1 of 2)

Inherits the full GQA suite (decode G∈{1,2,4,8} × non-mult `N_k` × d{64,128} × causal both ways;
idle-warp G=3 + multi-tile G=16; square reduction). `v8_gqa` confirms no regression.

In [ ]:
!python -m pytest tests/test_correctness.py -k "v8_gqa_ss or v8_gqa" -q


## 5. THE A/B — v8_gqa_ss vs Cut 1 (headline) and vs occ (layout-isolated), G-sweep

`v8_gqa` = Cut 1 (GEMV, FP32 smem). `v8_gqa_occ` = same GEMV but FP16 smem / higher occupancy — so
**ss vs occ isolates the LAYOUT** (both FP16 smem). A µs/tok drop on ss (esp. d=64) confirms the
reduction was the wall; a d=128 null points at smem-BW; flat %HBM despite faster µs/tok reframes the
floor as load latency.

In [ ]:
print('=== Cut 1: v8_gqa (GEMV, FP32 smem) ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== occ: v8_gqa_occ (GEMV, FP16 smem) — the layout-isolated baseline ===')
!python -m bench.harness --backend v8_gqa_occ --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== v8.7: v8_gqa_ss (score-stationary, FP16 smem) ===')
!python -m bench.harness --backend v8_gqa_ss --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32


## 6. Reclaim-SDPA-at-batch (G=8) — does removing the reduction move the µs/tok floor at B≥8?

In [ ]:
print('=== Cut 1: v8_gqa ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
print('\n=== occ: v8_gqa_occ ===')
!python -m bench.harness --backend v8_gqa_occ --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
print('\n=== v8.7: v8_gqa_ss ===')
!python -m bench.harness --backend v8_gqa_ss --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
